# Machine Learning Lab

## Student Information

- **Name:** Talha Abdullah Bangyal
- **Roll Number:** SU92-BAIFM-F25-002
- **Section:** BSAI-7C
- **Subject:** Machine Learning Lab
- **Lab:** 6

## Imports

In [2]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# Task 1 — KNN Concept & Distance

### (a) How KNN classifies an email
K-Nearest Neighbors (KNN) classifies a new email by comparing its features with previously labeled emails. It calculates the distance between the new email and the training emails, using features such as the number of links and exclamation marks. It selects the K nearest neighbors and uses a **majority vote** to choose the class. If most neighbors are Spam, the email is classified as Spam; otherwise, it is classified as Ham.

### (b) Euclidean distance between Email A and Email B

In [3]:
A = np.array([2, 1])  # (num_links, num_exclamations)
B = np.array([5, 4])

distance = np.linalg.norm(A - B)
print("Euclidean Distance:", round(distance, 3))

Euclidean Distance: 4.243


### (c) Small K or large K?
A small K makes KNN more sensitive to a single noisy or mislabeled training point because fewer neighbors participate in the majority vote, so one incorrect point can have more influence.

# Task 2 — Build the Confusion Matrix

Spam is the positive class. In the matrix with `labels=["Spam", "Ham"]`, rows represent actual labels and columns represent predicted labels.

In [22]:
# Actual labels from the 20 emails
actual = [
    "Spam", "Spam", "Ham", "Spam", "Ham",
    "Spam", "Ham", "Spam", "Ham", "Spam",
    "Ham", "Spam", "Spam", "Ham", "Ham",
    "Spam", "Ham", "Spam", "Spam", "Ham"
]

# Predicted labels from the 20 emails
predicted = [
    "Spam", "Spam", "Ham", "Ham", "Ham",
    "Spam", "Spam", "Spam", "Ham", "Spam",
    "Ham", "Ham", "Spam", "Ham", "Spam",
    "Spam", "Ham", "Ham", "Spam", "Ham"
]

cm = confusion_matrix(actual, predicted, labels=["Spam", "Ham"])
print("Confusion Matrix (rows=actual, columns=predicted):")
print(cm)

# Calculate True Positives (TP), False Negatives (FN), False Positives (FP), and True Negatives (TN)
TP = cm[0, 0]
FN = cm[0, 1]
FP = cm[1, 0]
TN = cm[1, 1]

print(f"TP = {TP}, TN = {TN}, FP = {FP}, FN = {FN}")

Confusion Matrix (rows=actual, columns=predicted):
[[8 3]
 [2 7]]
TP = 8, TN = 7, FP = 2, FN = 3


### (b) TP, TN, FP, and FN
- **TP (True Positive): 8** — Spam correctly predicted as Spam.
- **TN (True Negative): 7** — Ham correctly predicted as Ham.
- **FP (False Positive): 2** — Ham incorrectly predicted as Spam.
- **FN (False Negative): 3** — Spam incorrectly predicted as Ham.

### (c) Dangerous and minor mistakes
- **Most dangerous: False Negative (FN).** A malicious Spam email is classified as Ham and may reach the inbox, exposing the user or organization to phishing or malware.
- **Minor nuisance: False Positive (FP).** A legitimate Ham email is classified as Spam and may be blocked or sent to the spam folder, delaying a genuine message.

# Task 3 — Bayes' Theorem Calculation

Given: 30% of emails are Spam; “urgent” appears in 55% of Spam emails and 8% of Ham emails.

In [6]:
# Given probabilities
P_spam = 0.30
P_ham = 1 - P_spam
P_urgent_given_spam = 0.55
P_urgent_given_ham = 0.08

In [7]:
# (a) Law of total probability: P(urgent)
P_urgent = (
    P_urgent_given_spam * P_spam
    + P_urgent_given_ham * P_ham
)
print("P(urgent):", round(P_urgent, 3))

P(urgent): 0.221


In [8]:
# (b) Bayes' theorem: P(Spam | urgent)
P_spam_given_urgent = (P_urgent_given_spam * P_spam) / P_urgent
print("P(Spam | urgent):", round(P_spam_given_urgent, 3))

P(Spam | urgent): 0.747


In [9]:
# (c) Bayes' theorem: P(Ham | urgent)
P_ham_given_urgent = (P_urgent_given_ham * P_ham) / P_urgent
print("P(Ham | urgent):", round(P_ham_given_urgent, 3))

P(Ham | urgent): 0.253


### (d) Interpretation
The probability of Spam increases from the base rate of 30% to about 74.7% when the word “urgent” appears. This is because “urgent” appears in 55% of Spam emails but only 8% of Ham emails, so it is evidence associated more strongly with Spam. The probability of Ham given “urgent” is about 25.3%.

# Task 4 — Choosing K & the Majority Vote

The new email E has features `(num_links=3, num_exclamations=2)`.

In [10]:
# Store training emails as (name, features, label)
training_data = [
    ("T1", np.array([1, 1]), "Spam"),
    ("T2", np.array([2, 2]), "Ham"),
    ("T3", np.array([6, 5]), "Spam"),
    ("T4", np.array([4, 3]), "Spam"),
    ("T5", np.array([0, 0]), "Ham"),
    ("T6", np.array([5, 4]), "Spam"),
    ("T7", np.array([3, 1]), "Ham")
]

E = np.array([3, 2])

# Calculate each distance and sort from nearest to farthest
distances = []
for name, features, label in training_data:
    d = np.linalg.norm(E - features)
    distances.append((name, d, label))

distances.sort(key=lambda item: (item[1], item[0]))

print("Distances from nearest to farthest:")
for name, d, label in distances:
    print(f"{name}: Distance = {d:.3f}, Label = {label}")

Distances from nearest to farthest:
T2: Distance = 1.000, Label = Ham
T7: Distance = 1.000, Label = Ham
T4: Distance = 1.414, Label = Spam
T1: Distance = 2.236, Label = Spam
T6: Distance = 2.828, Label = Spam
T5: Distance = 3.606, Label = Ham
T3: Distance = 4.243, Label = Spam


In [11]:
# (b) Prediction for K = 3
K = 3
nearest_3 = distances[:K]
votes_3 = Counter(label for _, _, label in nearest_3)
prediction_k3 = votes_3.most_common(1)[0][0]

print("K=3 neighbors:")
for name, d, label in nearest_3:
    print(f"{name}: Distance = {d:.3f}, Label = {label}")
print("Prediction for K=3:", prediction_k3)

K=3 neighbors:
T2: Distance = 1.000, Label = Ham
T7: Distance = 1.000, Label = Ham
T4: Distance = 1.414, Label = Spam
Prediction for K=3: Ham


In [12]:
# (c) Prediction for K = 5
K = 5
nearest_5 = distances[:K]
votes_5 = Counter(label for _, _, label in nearest_5)
prediction_k5 = votes_5.most_common(1)[0][0]

print("K=5 neighbors:")
for name, d, label in nearest_5:
    print(f"{name}: Distance = {d:.3f}, Label = {label}")
print("Prediction for K=5:", prediction_k5)

K=5 neighbors:
T2: Distance = 1.000, Label = Ham
T7: Distance = 1.000, Label = Ham
T4: Distance = 1.414, Label = Spam
T1: Distance = 2.236, Label = Spam
T6: Distance = 2.828, Label = Spam
Prediction for K=5: Spam


### (d) Compare K = 3 and K = 5
With K = 3, the nearest neighbors contain two Ham emails and one Spam email, so the majority vote predicts **Ham**. With K = 5, the selected neighbors contain three Spam emails and two Ham emails, so the prediction changes to **Spam**.

# Task 5 — Compute the Metrics

Given TP = 96, FP = 6, FN = 14, and TN = 84.

In [13]:
TP, FP, FN, TN = 96, 6, 14, 84
total = TP + FP + FN + TN

accuracy = (TP + TN) / total
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1_score = 2 * precision * recall / (precision + recall)

print("Accuracy:", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1-score:", round(f1_score, 3))

Accuracy: 0.9
Precision: 0.941
Recall: 0.873
F1-score: 0.906


### (b) Which single metric could be misleading?
Accuracy alone could be misleading because it does not show the types of errors made by the filter. Recall is especially important in cybersecurity because a low Recall means more malicious emails may be missed, even when overall Accuracy looks high.

# Task 6 — Why Is It Called “Naïve”?

### (a) Independence assumption
Naïve Bayes is called “naïve” because it assumes that words are conditionally independent of one another once the email class (Spam or Ham) is known. Real language does not always work this way because words often appear together in meaningful phrases.

### (b) Example
“Free” and “prize” may frequently appear together in spam emails. Their appearances can be related, so the independence assumption may not hold.

### (c) Why it can still work
Naïve Bayes is fast, computationally efficient, and often works well with high-dimensional text data. Even if its independence assumption is imperfect, word frequencies can still provide useful signals for spam classification.

# Task 7 — KNN Using the Provided CSV

The supplied file is named `email_features_csv.csv`. Keep it in the same folder as this notebook. This section uses the actual CSV data.

In [19]:
# Load the provided CSV file
csv_path = "email_features_csv.csv"
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst five rows:")
display(df.head())

# Validate required columns
required_columns = ["num_links", "num_exclamations", "has_attachment", "label"]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Select features and target
X = df[["num_links", "num_exclamations", "has_attachment"]]
y = df["label"]

# Split 75% training and 25% testing, reproducibly
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

Dataset shape: (300, 4)
Columns: ['num_links', 'num_exclamations', 'has_attachment', 'label']

First five rows:


,num_links,num_exclamations,has_attachment,label
0,5,3,0,Spam
1,4,3,1,Spam
2,4,3,0,Spam
3,4,5,0,Spam
4,2,3,0,Spam



Training samples: 225
Testing samples: 75


In [18]:
# Fit KNN with K = 5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Predict on the test set and calculate accuracy
y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy, 4))

Model Accuracy: 0.6133


### (d) What is `random_state`?
`random_state=42` makes the train/test split reproducible, so the same rows are selected each time the notebook runs; without it, the split and the resulting accuracy may vary.

### (e) Is accuracy alone enough?
No. An accuracy of 0.82 means 82% of test examples were classified correctly, but it does not show how many malicious emails were missed or how many legitimate emails were incorrectly flagged. Precision, Recall, and the confusion matrix are also needed to assess a spam filter.

# Task 8 — Precision vs. Recall Trade-off

Filter A: TP = 90, FP = 2, FN = 20. Filter B: TP = 108, FP = 25, FN = 2.

In [20]:
# Filter A
TP_A, FP_A, FN_A = 90, 2, 20
precision_A = TP_A / (TP_A + FP_A)
recall_A = TP_A / (TP_A + FN_A)

# Filter B
TP_B, FP_B, FN_B = 108, 25, 2
precision_B = TP_B / (TP_B + FP_B)
recall_B = TP_B / (TP_B + FN_B)

print("Filter A")
print("Precision:", round(precision_A, 3))
print("Recall:", round(recall_A, 3))

print("\nFilter B")
print("Precision:", round(precision_B, 3))
print("Recall:", round(recall_B, 3))

Filter A
Precision: 0.978
Recall: 0.818

Filter B
Precision: 0.812
Recall: 0.982


### (b) Business interpretation
Filter A has Precision of about 97.8%, so almost all emails it flags are actually Spam and it produces few false alarms. However, its Recall is about 81.8%, meaning it misses around 18.2% of actual Spam emails. Filter B has lower Precision (about 81.2%), so it creates more false alarms, but its Recall of about 98.2% means it detects nearly all actual Spam emails.

### (c) Which filter should the firm prefer?
The firm should prefer **Filter B** because a missed phishing email can create a serious security risk, while a false alarm usually causes inconvenience. Filter B has much higher Recall and misses far fewer malicious emails, although the company should manage the additional false positives.